# Lab 1: Run the Flower PyTorch Quickstart with 10 Clients
**Course:** Fundamentals & Architectures of Federated Learning
**Lecturer:** Ts. Dr. Mohammad Kamrul Hasan, Centre for Cyber Security, FTSM, Universiti Kebangsaan Malaysia (UKM)

In this lab you will train an image classifier on **CIFAR-10** with **federated learning (FL)**: 10 simulated clients each train on their own data partition, and a server combines their models with **FedAvg**. No client ever shares its images.

**What you will do**
1. Install Flower and load CIFAR-10 split into 10 client partitions
2. Define a small CNN, plus local `train` and `test` functions
3. Write the **ClientApp** (client side) and **ServerApp** (server side, FedAvg)
4. Run a 10-client simulation and plot accuracy per round
5. Answer the reflection questions; optional extensions lead into Labs 2 to 4

> **Runtime:** works on CPU (about 5 to 10 minutes for 5 rounds). For speed, choose *Runtime → Change runtime type → GPU* in Colab.

> Code uses Flower's current **Message API** (`ClientApp` / `ServerApp` / `strategy.start`), checked against Flower 1.44.

## Step 1 · Install Flower
Run once per Colab session. PyTorch and torchvision are already installed in Colab.

In [ ]:
!pip install -q "flwr[simulation]==1.44.0" "flwr-datasets[vision]==0.6.1"
import flwr, flwr_datasets
print("Flower", flwr.__version__, "| Flower Datasets", flwr_datasets.__version__)

## Step 2 · Imports and lab settings
These constants are the FL hyper-parameters from the lecture: **K** clients, **T** rounds, **C** fraction per round, **E** local epochs, **B** batch size and learning rate **η**.

In [ ]:
from collections import OrderedDict
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

from flwr.app import ArrayRecord, ConfigRecord, Context, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner

NUM_CLIENTS   = 10     # K: number of clients (virtual SuperNodes)
NUM_ROUNDS    = 5      # T: communication rounds
FRACTION_TRAIN = 0.5   # C: fraction of clients sampled each round
LOCAL_EPOCHS  = 1      # E: local epochs per round
BATCH_SIZE    = 32     # B
LEARNING_RATE = 0.01   # η (client learning rate)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on", DEVICE)

## Step 3 · Load CIFAR-10 and split it across 10 clients
`FederatedDataset` downloads CIFAR-10 once and splits the **training set** into 10 partitions. Each client sees only its own partition and keeps 20% of it for local validation. The official **test set** is held by the server for centralized evaluation.

We start with an **IID** split (`IidPartitioner`): every client gets a similar mix of classes. Lab 2 switches to a non-IID Dirichlet split.

In [ ]:
fds = None  # cached per process so each client does not re-download the data

pytorch_transforms = Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

def apply_transforms(batch):
    batch["img"] = [pytorch_transforms(img) for img in batch["img"]]
    return batch

def load_data(partition_id: int, num_partitions: int):
    """Return (trainloader, valloader) for one client's partition."""
    global fds
    if fds is None:
        partitioner = IidPartitioner(num_partitions=num_partitions)
        fds = FederatedDataset(dataset="uoft-cs/cifar10", partitioners={"train": partitioner})
    partition = fds.load_partition(partition_id)
    split = partition.train_test_split(test_size=0.2, seed=42)
    split = split.with_transform(apply_transforms)
    trainloader = DataLoader(split["train"], batch_size=BATCH_SIZE, shuffle=True)
    valloader = DataLoader(split["test"], batch_size=BATCH_SIZE)
    return trainloader, valloader

def load_centralized_testset():
    """The server's held-out test set (10,000 CIFAR-10 test images)."""
    testset = FederatedDataset(dataset="uoft-cs/cifar10", partitioners={"train": 1}).load_split("test")
    return DataLoader(testset.with_transform(apply_transforms), batch_size=128)

# Quick look: how many training images does each client hold?
sizes = []
for pid in range(NUM_CLIENTS):
    tl, vl = load_data(pid, NUM_CLIENTS)
    sizes.append(len(tl.dataset))
print("Training images per client:", sizes)

In [ ]:
# Optional: see the label mix of each client (should look balanced for IID)
from flwr_datasets.visualization import plot_label_distributions
fig, ax, df = plot_label_distributions(fds.partitioners["train"], label_name="label",
                                       plot_type="bar", size_unit="percent", legend=True,
                                       title="Label distribution per client (IID)")

## Step 4 · Model, local training and testing
A small CNN (the same as the Flower quickstart). `train` is the client's local SGD, exactly the *ClientUpdate* of Algorithm 1 in the slides; `test` measures loss and accuracy.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16 * 5 * 5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def train(net, trainloader, epochs, lr, device):
    """Local SGD on one client's data. Returns the average training loss."""
    net.to(device).train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    running_loss, steps = 0.0, 0
    for _ in range(epochs):
        for batch in trainloader:
            images, labels = batch["img"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            loss = criterion(net(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item(); steps += 1
    return running_loss / max(steps, 1)

def test(net, testloader, device):
    """Returns (average loss, accuracy)."""
    net.to(device).eval()
    criterion = nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for batch in testloader:
            images, labels = batch["img"].to(device), batch["label"].to(device)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return loss / len(testloader), correct / len(testloader.dataset)

## Step 5 · The ClientApp (runs on every client)
Each client receives the global model **wᵗ** inside a `Message`, trains locally, and replies with its new weights **wₖ** and its number of examples **nₖ**. The key `"num-examples"` is what FedAvg uses to weight the average (pₖ = nₖ / n).

In [ ]:
client_app = ClientApp()

@client_app.train()
def client_train(msg: Message, context: Context):
    # 1. Load the global model w^t sent by the server
    model = Net()
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict())
    # 2. Load this client's own data partition
    pid = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    trainloader, _ = load_data(pid, num_partitions)
    # 3. Local training (E epochs of SGD)
    lr = msg.content["config"]["lr"]
    train_loss = train(model, trainloader, LOCAL_EPOCHS, lr, DEVICE)
    # 4. Reply with w_k and n_k (never the data itself)
    content = RecordDict({
        "arrays": ArrayRecord(torch_state_dict=model.state_dict()),
        "metrics": MetricRecord({"train_loss": train_loss,
                                 "num-examples": len(trainloader.dataset)}),
    })
    return Message(content=content, reply_to=msg)

@client_app.evaluate()
def client_evaluate(msg: Message, context: Context):
    # Federated evaluation: test the global model on this client's validation split
    model = Net()
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict())
    pid = context.node_config["partition-id"]
    _, valloader = load_data(pid, context.node_config["num-partitions"])
    loss, acc = test(model, valloader, DEVICE)
    content = RecordDict({"metrics": MetricRecord({"eval_loss": loss, "eval_acc": acc,
                                                   "num-examples": len(valloader.dataset)})})
    return Message(content=content, reply_to=msg)

## Step 6 · The ServerApp (FedAvg)
`FedAvg(fraction_train=0.5)` samples half of the 10 clients each round (C = 0.5) and averages their models weighted by `num-examples`. `central_evaluate` tests the global model on the server's test set after every round.

In [ ]:
HISTORY = {}   # filled by the server so we can plot afterwards
server_app = ServerApp()

@server_app.main()
def server_main(grid: Grid, context: Context) -> None:
    testloader = load_centralized_testset()

    def central_evaluate(server_round: int, arrays: ArrayRecord) -> MetricRecord:
        model = Net()
        model.load_state_dict(arrays.to_torch_state_dict())
        loss, acc = test(model, testloader, DEVICE)
        return MetricRecord({"loss": loss, "accuracy": acc})

    strategy = FedAvg(fraction_train=FRACTION_TRAIN, fraction_evaluate=0.5)
    result = strategy.start(
        grid=grid,
        initial_arrays=ArrayRecord(torch_state_dict=Net().state_dict()),   # w^0
        num_rounds=NUM_ROUNDS,
        train_config=ConfigRecord({"lr": LEARNING_RATE}),
        evaluate_fn=central_evaluate,
    )
    HISTORY["result"] = result
    torch.save(result.arrays.to_torch_state_dict(), "final_global_model.pt")
    print("Saved final global model to final_global_model.pt")

## Step 7 · Run the simulation with 10 clients
`run_simulation` starts 10 virtual clients (SuperNodes) on this machine. In the log, each round shows: `configure_train: Sampled 5 nodes (out of 10)`, `aggregate_train: Received 5 results and 0 failures`, then the global evaluation.

*(Flower prints a note that `run_simulation` is deprecated in favour of the `flwr run` CLI. It still works and is the simplest option in a notebook; the CLI is shown in the lecture slides.)*

In [ ]:
backend_config = {"client_resources": {"num_cpus": 1,
                                      "num_gpus": 0.1 if DEVICE.type == "cuda" else 0.0}}

run_simulation(server_app=server_app, client_app=client_app,
               num_supernodes=NUM_CLIENTS, backend_config=backend_config)

## Step 8 · Plot the results

In [ ]:
result = HISTORY["result"]
central = result.evaluate_metrics_serverapp      # {round: MetricRecord}
rounds = sorted(central.keys())
acc = [central[r]["accuracy"] * 100 for r in rounds]
loss = [central[r]["loss"] for r in rounds]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rounds, acc, marker="o", color="#2F6E62")
axes[0].set(title="Global test accuracy", xlabel="Round (0 = initial model)", ylabel="Accuracy (%)")
axes[1].plot(rounds, loss, marker="s", linestyle="--", color="#D49A3A")
axes[1].set(title="Global test loss", xlabel="Round", ylabel="Cross-entropy loss")
for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Round | accuracy")
for r, a in zip(rounds, acc):
    print(f"{r:>5} | {a:5.1f}%")
print("Federated (client-side) evaluation:", dict(result.evaluate_metrics_clientapp))

## Step 9 · Reflection questions
1. How many clients trained in each round, and why? (Hint: `FRACTION_TRAIN`.)
2. Which line in `client_train` sends the information FedAvg needs to weight clients? What would happen without it?
3. Compare the server's **centralized** accuracy with the clients' **federated** evaluation. Why can they differ?
4. Change `NUM_ROUNDS` to 10 and `FRACTION_TRAIN` to 1.0. How do accuracy and run time change?
5. Change `LOCAL_EPOCHS` to 5. Does accuracy per round improve? What is the cost? (Recall the FedSGD vs FedAvg slide.)

### What to record for your lab report
Flower version · number of clients · rounds · fraction-train · local epochs · learning rate · final accuracy · the accuracy plot.

## Optional extensions (lead into Labs 2 to 4)
**Lab 2: non-IID data.** In `load_data`, replace the partitioner:
```python
from flwr_datasets.partitioner import DirichletPartitioner
partitioner = DirichletPartitioner(num_partitions=num_partitions, partition_by="label",
                                   alpha=0.1, min_partition_size=10, seed=42)
```
Restart the runtime (so `fds` is rebuilt), re-run, and compare accuracy with the IID run.

**Lab 3: FedProx.** Use `from flwr.serverapp.strategy import FedProx` and `FedProx(fraction_train=..., proximal_mu=0.1)`; in `train`, add `(mu / 2) * ||w − w_global||²` to the loss, reading `mu = msg.content["config"].get("proximal-mu", 0.0)`.

**Lab 4: differential privacy.** Wrap the strategy with `DifferentialPrivacyClientSideFixedClipping(strategy, noise_multiplier=1.0, clipping_norm=1.0, num_sampled_clients=5)` and create the client with `ClientApp(mods=[fixedclipping_mod])` (from `flwr.clientapp.mod`).

## Troubleshooting
| Symptom | Fix |
|---|---|
| `Received 0 results and N failures` | A client crashed: scroll up for the Python traceback in the client log |
| Ray / out-of-memory errors | Lower `num_cpus`/`num_gpus` per client, or use fewer clients |
| First run is slow | CIFAR-10 is downloading; it is cached afterwards |
| Accuracy stays near 10% | Check `load_state_dict` and try a smaller learning rate |
| Changes to the partitioner have no effect | Restart the runtime so the cached `fds` is rebuilt |